In [6]:
#lOAD PACKAGES
import pandas as pd
import matplotlib.pyplot as plt
import os
import plotly.express as px
# Import the processing module from the same folder
from processing import load_solutions, add_kwargs_as_indices, combine_solutions, read_parquet_and_convert, add_fields
# import processing 
# from pivottablejs import pivot_ui
G_save = True

In [7]:
def create_envelope(s_ed, s_uc, group_by = ['configuration', 'µ', 'iteration', 'day', 'hour,', 'r_id']):
    # Copy the relevant columns from s_ed['storage']
    envelope = s_ed['storage'][group_by + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].copy()

    # Perform the first left join
    envelope = envelope.merge(
        s_uc['storage'][[col for col in group_by if col != 'iteration'] + ['SOE_MWh', 'envelope_up_MWh', 'envelope_down_MWh']].rename(
            columns={'SOE_MWh': 'SOE_DA_MWh', 'envelope_up_MWh': 'envelope_up_DA_MWh', 'envelope_down_MWh': 'envelope_down_DA_MWh'}
        ),
        on=[col for col in group_by if col != 'iteration'],
        how='left'
    )
    
    # Perform the second left join
    envelope = envelope.merge(
        s_uc['storage_parameters'][['r_id', 'SOE_max_MWh', 'initial_energy_proportion']].drop_duplicates(),
        on='r_id',
        how='left'
    )
    
    # Calculate initial state of energy (SOE) based on maximum SOE and initial energy proportion
    envelope['SOE_0_MWh'] = envelope['SOE_max_MWh'] * envelope['initial_energy_proportion']
    envelope['SOC'] = envelope['SOE_MWh'] / envelope['SOE_max_MWh'] 
    # Group by day, configuration, and resource ID, and get the last entry for each group

    return envelope

In [8]:

ss = [
    # {'solution_folder': f"RTS-GMLC_v5.2.2s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2z", 'model_type' : 'envelope'}, # last opertional
    # {'solution_folder': f"RTS-GMLC_v5.2.2.1s", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.3.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v5.2.2.4.1z", 'model_type' : 'envelope'},
    # {'solution_folder': f"RTS-GMLC_v6.2.2s", 'model_type' : 'e-reserve'},
    {'solution_folder': f"RTS-GMLC_v9.0s", 'VLGEN': 30, 'model_type' : 'envelope'},
    {'solution_folder': f"RTS-GMLC_v10.0s", 'VLGEN': 30, 'model_type' : 'e-reserve'},
    # {'solution_folder': f"RTS-GMLC_v6.2.3s", 'model_type' : 'e-reserve'}
    ]

s_uc = []
s_ed = []
gcd_KPI_adequacy = []
gcdi_KPI_adequacy = []
for sol in ss:
    # ρ = sol['ρ']
    s = sol['solution_folder']
    s_uc_name = 's_suc' if sol['model_type'] == 'stochastic' else 's_uc'
    # s_uc_ = load_solutions(s_uc_name, os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_ed_ = load_solutions("s_ed", os.path.join("..", "output", s), [7], model_type = sol['model_type'], ρ=ρ, solution_id = s)
    # s_uc.append(s_uc_)
    # s_ed.append(s_ed_)

    gcd_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcd_KPI_adequacy.parquet"))
    gcd_KPI_adequacy_ = add_fields(gcd_KPI_adequacy_, model_type = sol['model_type'], solution_id = s) 

    gcdi_KPI_adequacy_ = read_parquet_and_convert( os.path.join("..", "output", s, "all_gcdi_KPI_adequacy.parquet"))
    gcdi_KPI_adequacy_ = add_fields(gcdi_KPI_adequacy_, model_type = sol['model_type'], solution_id = s)

    gcd_KPI_adequacy.append(gcd_KPI_adequacy_)
    gcdi_KPI_adequacy.append(gcdi_KPI_adequacy_)

# s_uc = combine_solutions(s_uc)
# s_ed = combine_solutions(s_ed)
gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)

if 'µ' in gcdi_KPI_adequacy.columns: 
#         # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
    gcdi_KPI_adequacy['model_type'] = gcdi_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)
    gcd_KPI_adequacy['model_type'] = gcd_KPI_adequacy.apply(lambda x: 'conservative' if ('envelope' in x['model_type']) & (x['µ'] == 1) else x['model_type'], axis=1)




../output/RTS-GMLC_v9.0s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v9.0s/all_gcdi_KPI_adequacy.parquet
../output/RTS-GMLC_v10.0s/all_gcd_KPI_adequacy.parquet
../output/RTS-GMLC_v10.0s/all_gcdi_KPI_adequacy.parquet


/tmp/ipykernel_1632/2963992833.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gcd_KPI_adequacy = pd.concat(gcd_KPI_adequacy)
/tmp/ipykernel_1632/2963992833.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gcdi_KPI_adequacy = pd.concat(gcdi_KPI_adequacy)


In [9]:
gcdi_KPI_adequacy['model_type'].unique()

array(['envelope', 'conservative', 'e-reserve'], dtype=object)

In [10]:
if G_save:
    out= gcd_KPI_adequacy.copy()
    if 'µ' in out.columns: 
        out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)
        # out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: ((x[0] =='envelope')*(x[1]==1)*'conservatice' + x[0]), axis = 1)

    renames = {'µ': 'mu', 'ρ' : 'rho'}
    renames = {k: v for k, v in renames.items() if k in out.columns}
    out.rename(columns = renames, inplace = True)
    out.to_csv('gcd_KPI_adequacy.csv', index=False)
    gcdi_KPI_adequacy.rename(columns = renames, inplace = True)
    gcdi_KPI_adequacy.reset_index().to_csv('gcdi_KPI_adequacy.csv', index=False)

/tmp/ipykernel_1632/3797213802.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  out['non-conservative'] = out[['model_type', 'µ']].apply(lambda x: (x[0] !='envelope') + (x[0] =='envelope')*(x[1]<1), axis = 1)


In [11]:
gcdi_KPI_adequacy

,model_type,solution_id,configuration,day,iteration,LLD_h,ENS_MWh,input_load_MWh,CURD_h,CUR_MWh,...,slack_energy_reserve_down_uc_MWh,required_energy_reserve_up_uc_MWh,slack_energy_reserve_up_uc_MWh,required_energy_reserve_down_uc_MWh,energy_reserve_down_uc_MWh,energy_reserve_up_uc_MWh,thermal_energy_reserve_down_uc_MWh,thermal_energy_reserve_up_uc_MWh,storage_energy_reserve_down_uc_MWh,storage_energy_reserve_up_uc_MWh
0,envelope,RTS-GMLC_v9.0s,base_ramp_storage_envelopes_up_0_54_dn_0_54,1,demand_1,0,3.899719e-17,45565.932067,0,2.442491e-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,RTS-GMLC_v9.0s,base_ramp_storage_envelopes_up_1_dn_1,1,demand_1,0,5.045072e-13,45565.932067,0,2.442491e-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,conservative,RTS-GMLC_v9.0s,base_ramp_storage_envelopes_up_1_dn_1,2,demand_1,0,7.317879e-13,42264.410558,0,1.145750e-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,envelope,RTS-GMLC_v9.0s,base_ramp_storage_envelopes_up_0_49_dn_0_49,2,demand_1,0,2.137000e-13,42264.410558,0,1.145750e-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,envelope,RTS-GMLC_v9.0s,base_ramp_storage_envelopes_up_0_63_dn_0_63,3,demand_1,0,8.156701e-14,39266.476180,0,8.881784e-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,e-reserve,RTS-GMLC_v10.0s,base_ramp_storage_envelopes_up_1_dn_1,126,demand_1,0,1.131645e-15,72472.018898,0,4.796163e-14,...,1.396875e-08,1.194221e+06,1.396875e-08,1.543075e+06,1.543086e+06,1.194249e+06,468386.416463,235546.883630,1.074699e+06,9.587024e+05
126,e-reserve,RTS-GMLC_v10.0s,base_ramp_storage_envelopes_up_1_dn_1,127,demand_1,0,4.798289e-14,58685.646387,0,-4.973799e-14,...,1.142268e-08,1.130845e+06,1.142265e-08,1.595435e+06,1.595439e+06,1.130852e+06,506866.630711,201709.446109,1.088573e+06,9.291429e+05
127,e-reserve,RTS-GMLC_v10.0s,base_ramp_storage_envelopes_up_1_dn_1,128,demand_1,0,5.497320e-13,53049.872677,0,-1.048051e-13,...,2.376174e-08,1.339982e+06,2.376174e-08,1.603521e+06,1.603528e+06,1.340010e+06,402261.218870,202351.629025,1.201266e+06,1.137658e+06
128,e-reserve,RTS-GMLC_v10.0s,base_ramp_storage_envelopes_up_1_dn_1,129,demand_1,0,2.353985e-13,69391.058969,0,-3.907985e-14,...,8.069263e-09,8.974920e+05,8.056243e-09,1.382371e+06,1.382374e+06,8.974967e+05,471646.311803,175474.560699,9.107276e+05,7.220222e+05
